# **Imports and Initialization**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Installing Unsloth**

In [2]:
%%writefile /dev/null
# Cell 1 — Install Unsloth and dependencies

# Step 1: Install Unsloth with the correct CUDA-aware wheel
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Step 2: Install xformers for memory-efficient attention
!pip install --no-deps xformers trl peft accelerate bitsandbytes

Overwriting /dev/null


# **Loading Model and Tokenizer**

In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-dygj5s9h/unsloth_4546f00e0dcf46cea637c4df1e6f651a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-dygj5s9h/unsloth_4546f00e0dcf46cea637c4df1e6f651a
  Resolved https://github.com/unslothai/unsloth.git to commit 1798ebe3f30281978abdc7e409bd91a11a1faa8a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 97.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 13.6 MB/s eta 0:00:00


In [3]:
!pip show unsloth

Name: unsloth
Version: 2026.5.10
Summary: 2-5X faster training, reinforcement learning & finetuning
Home-page: https://unsloth.ai
Author: Unsloth AI team
Author-email: info@unsloth.ai
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: nest-asyncio, pydantic, pyyaml, typer
Required-by: 


In [6]:
from unsloth import FastLanguageModel
import torch

# ── Configuration ──────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048   # Must match your Stage 3 length filter
DTYPE = None            # Auto-detect: float16 on T4
LOAD_IN_4BIT = True     # 4-bit quantisation to fit on T4

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

# ── Load ───────────────────────────────────────────────────────
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

print("Model loaded ✓")
print(f"  Parameters : {model.num_parameters() / 1e9:.2f}B")
print(f"  Device     : {next(model.parameters()).device}")
print(f"  Dtype      : {next(model.parameters()).dtype}")

==((====))==  Unsloth 2026.5.10: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

unsloth/Qwen2.5-3B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Model loaded ✓
  Parameters : 3.09B
  Device     : cuda:0
  Dtype      : torch.float16


# **Configuring LoRA Adapters**

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,
    lora_alpha          = 32,
    lora_dropout        = 0,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    bias                = "none",   # Don't train bias terms — saves VRAM
    use_gradient_checkpointing = "unsloth",
    random_state        = 42,
)

# Summarise what's actually trainable
model.print_trainable_parameters()

Unsloth 2026.5.10 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


# **Load & format train.jsonl / val.jsonl**

In [8]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# ── Apply Qwen2.5 chat template to tokeniser ───────────────────
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

# ── Formatting function ────────────────────────────────────────
EOS_TOKEN = tokenizer.eos_token

def format_example(example):
    """Render the already-structured messages into a ChatML string."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize              = False,
        add_generation_prompt = False,
    )
    return {"text": text + EOS_TOKEN}

# ── Load your JSONL files ──────────────────────────────────────
DATA_DIR = "/content/drive/MyDrive/SISSA_Thesis/data/final"

dataset = load_dataset(
    "json",
    data_files = {
        "train" : f"{DATA_DIR}/train.jsonl",
        "val"   : f"{DATA_DIR}/val.jsonl",
    }
)

# ── Apply formatting ───────────────────────────────────────────
dataset = dataset.map(format_example, batched=False)

# ── Sanity check ───────────────────────────────────────────────
print(f"Train examples : {len(dataset['train'])}")
print(f"Val examples   : {len(dataset['val'])}")
print("\n── Sample (first training example) ──")
print(dataset["train"][0]["text"][:600], "...")

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/223 [00:00<?, ? examples/s]

Train examples : 2001
Val examples   : 223

── Sample (first training example) ──
<|im_start|>system
You are an expert in supersymmetric gauge theories and 3d mirror symmetry. Answer questions accurately and formally, using precise physical and mathematical language.<|im_end|>
<|im_start|>user
Explain how the structure of the scalar potential in supersymmetric gauge theories leads to the formation of a moduli space of vacua.<|im_end|>
<|im_start|>assistant
A moduli space of vacua emerges in a supersymmetric gauge theory when the scalar potential admits continuous families of zero-energy configurations. Since the potential is constructed such that it is manifestly non-negati ...


# **SFFT Hyperparameters**

In [10]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model     = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset  = dataset["val"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,        # Parallel tokenisation workers

    args = TrainingArguments(
        # ── Batch size ──────────────────────────────────────────
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch = 8

        # ── Learning rate ────────────────────────────────────────
        learning_rate        = 2e-4,
        lr_scheduler_type    = "cosine",
        warmup_ratio         = 0.05,       # Warm up for 5% of steps

        # ── Duration ─────────────────────────────────────────────
        num_train_epochs     = 3,          # 3 passes over your 2001 examples

        # ── Precision ────────────────────────────────────────────
        fp16 = not is_bfloat16_supported(),   # float16 on T4
        bf16 =     is_bfloat16_supported(),   # bfloat16 on newer GPUs

        # ── Logging & evaluation ─────────────────────────────────
        logging_steps        = 20,         # Print loss every 20 steps
        eval_strategy        = "steps",
        eval_steps           = 100,        # Evaluate on val set every 100 steps
        save_strategy        = "steps",
        save_steps           = 100,
        save_total_limit     = 2,          # Keep only 2 checkpoints on disk

        # ── Reproducibility & output ─────────────────────────────
        seed          = 42,
        output_dir    = "/content/drive/MyDrive/SISSA_Thesis/finetune/checkpoints",
        report_to     = "none",            # Disable wandb/tensorboard for now
    ),
)

print("Trainer configured ✓")
print(f"  Training examples   : {len(dataset['train'])}")
print(f"  Steps per epoch     : {len(dataset['train']) // 8}")
print(f"  Total steps         : {len(dataset['train']) // 8 * 3}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2001 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/223 [00:00<?, ? examples/s]

Trainer configured ✓
  Training examples   : 2001
  Steps per epoch     : 250
  Total steps         : 750


# **Run Training**

In [11]:
import torch

# ── VRAM snapshot before training ─────────────────────────────
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU: {gpu_stats.name}")
print(f"Total VRAM      : {max_memory} GB")
print(f"Reserved (start): {start_gpu_memory} GB")
print()

# ── Train ──────────────────────────────────────────────────────
trainer_stats = trainer.train()

# ── VRAM snapshot after training ──────────────────────────────
end_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"\nPeak VRAM used  : {end_gpu_memory} GB / {max_memory} GB")
print(f"Training time   : {trainer_stats.metrics['train_runtime']:.0f}s "
      f"({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Final train loss: {trainer_stats.metrics['train_loss']:.4f}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4
Total VRAM      : 14.563 GB
Reserved (start): 4.041 GB



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,001 | Num Epochs = 3 | Total steps = 753
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,1.170997,1.162507
200,1.119730,1.084718
300,0.914152,1.056770
400,0.928230,1.019505
500,0.902516,0.994248
600,0.680093,1.038971
700,0.680094,1.033499
753,0.693855,1.033047


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


Peak VRAM used  : 6.389 GB / 14.563 GB
Training time   : 2134s (35.6 min)
Final train loss: 0.9551


# **Saving the Model**

In [12]:
SAVE_DIR = "/content/drive/MyDrive/SISSA_Thesis/finetune"

# ── Step 1: Save LoRA adapters only ───────────────────────────
# Fast (~seconds). These are the trained delta weights only.
# To use later: load base model + apply these adapters.
model.save_pretrained(f"{SAVE_DIR}/lora_adapters")
tokenizer.save_pretrained(f"{SAVE_DIR}/lora_adapters")
print("LoRA adapters saved ✓")

# ── Step 2: Save merged model  ──────
# Merges adapters into base weights. Slower (~minutes) and larger (~6GB).
# Useful for inference without Unsloth, or uploading to HuggingFace.
print("\nMerging and saving full model (this takes a few minutes)...")

model.save_pretrained_merged(
    f"{SAVE_DIR}/merged_model",
    tokenizer,
    save_method = "merged_16bit",  # Full precision merged weights
)
print("Merged model saved ✓")

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SISSA_Thesis/finetune/lora_adapters/tokenizer_config.json.


LoRA adapters saved ✓

Merging and saving full model (this takes a few minutes)...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/SISSA_Thesis/finetune/merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [01:51<00:00, 55.63s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:07<00:00, 93.94s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/SISSA_Thesis/finetune/merged_model`
Merged model saved ✓


In [16]:
from huggingface_hub import login
from google.colab import userdata

# ── Authenticate ───────────────────────────────────────────────
login(token=userdata.get("HF_TOKEN"))

In [18]:
from huggingface_hub import HfApi
api = HfApi()

# Create repos first (do this once)
api.create_repo("anantshri1/qwen2.5-3b-mirror-symmetry", exist_ok=True)

RepoUrl('https://huggingface.co/anantshri1/qwen2.5-3b-mirror-symmetry', endpoint='https://huggingface.co', repo_type='model', repo_id='anantshri1/qwen2.5-3b-mirror-symmetry')

In [19]:
api.upload_folder(
    folder_path="/content/drive/MyDrive/SISSA_Thesis/finetune/lora_adapters",
    repo_id="anantshri1/qwen2.5-3b-mirror-symmetry",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...a_adapters/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

  ...adapter_model.safetensors:   0%|          |  559kB /  120MB            

CommitInfo(commit_url='https://huggingface.co/anantshri1/qwen2.5-3b-mirror-symmetry/commit/d4391cac109df45af344dc9fd3cd8c0f433f72a6', commit_message='Upload folder using huggingface_hub', commit_description='', oid='d4391cac109df45af344dc9fd3cd8c0f433f72a6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/anantshri1/qwen2.5-3b-mirror-symmetry', endpoint='https://huggingface.co', repo_type='model', repo_id='anantshri1/qwen2.5-3b-mirror-symmetry'), pr_revision=None, pr_num=None)

In [20]:
from unsloth.chat_templates import get_chat_template

# ── Switch model to inference mode ────────────────────────────
# This disables dropout and other training-specific behaviour
FastLanguageModel.for_inference(model)

# ── Helper function ───────────────────────────────────────────
def ask(question, max_new_tokens=300):
    messages = [
        {
            "role": "system",
            "content": "You are an expert in supersymmetric gauge theories "
                       "and 3d mirror symmetry. Answer questions accurately "
                       "and formally, using precise physical and mathematical language."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize               = True,
        add_generation_prompt  = True,   # Adds the assistant header to prompt generation
        return_tensors         = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids        = inputs,
        max_new_tokens   = max_new_tokens,
        temperature      = 0.7,
        top_p            = 0.9,
        use_cache        = True,
    )

    # Decode only the newly generated tokens (skip the prompt)
    response = tokenizer.decode(
        outputs[0][inputs.shape[1]:],
        skip_special_tokens = True
    )
    return response

# ── Test questions ────────────────────────────────────────────
questions = [
    # Something directly from your thesis domain
    "What is the role of the FI parameter in the mirror symmetry map for "
    "a U(1) theory with one charged hypermultiplet?",

    # Something requiring reasoning across concepts
    "Why does the Higgs branch of a 3d N=4 gauge theory become the Coulomb "
    "branch of its mirror?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {ask(q)}")
    print("\n" + "─"*60 + "\n")

Q: What is the role of the FI parameter in the mirror symmetry map for a U(1) theory with one charged hypermultiplet?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

A: In the context of the mirror symmetry map for a U(1) theory containing one charged hypermultiplet, the FI (Fayet–Iliopoulos) parameter serves as the duality moving counterpart to the real mass parameters $\pm m_H$. This parameter acts as a key input when identifying the specific column in the planar quiver that corresponds to the dual description of the theory.

────────────────────────────────────────────────────────────

Q: Why does the Higgs branch of a 3d N=4 gauge theory become the Coulomb branch of its mirror?
A: The transformation from the Higgs branch to the Coulomb branch in the mirror theory is dictated by the specific choice of fugacities in the partition function. In the original 3d N=4 theory, assigning large VEVs to real scalars along the Higgs branch corresponds to taking a large FI limit for the gauge group associated with that branch. When this same process is applied to the mirror theory, the analogous gauge group—which was originally coupled—effectively 'freezes' 